---
# **LAB 1 - Intro to Numba**
---

# ▶️ Google Colaboratory (colab)

[Colaboratory](https://research.google.com/colaboratory/faq.html) (or Colab) is a **free research tool** from *Google* for machine learning education and research built on top of [Jupyter Notebook](https://jupyter.org/). It requires no setup and runs entirely in the **cloud**. In Google Colab you can write, execute, save and share your Jupiter Notebooks. You access powerful computing resources like **TPUs** and **GPUs** all for free through your browser. All major Python libraries like **Tensorflow**, **Scikit-learn**, **PyTorch**, **Pandas**, etc. are pre-installed. Google Colab requires no configuration, you only need a **Google Account** and then you are good to go. Your notebooks are stored in your **Google Drive**, or can be loaded from **GitHub**. Colab notebooks can be shared just as you would with Google Docs or Sheets. Simply click the Share button at the top right of any Colab notebook, or follow these Google Drive file sharing instructions.




### Notebook rules

Some basic notebook rules:


1.   Click inside a cell with code and press SHIFT+ENTER (or click "PLAY" button) to execute it.
2.   Re-executing a cell will reset it (any input will be lost).
3.   Execute cells TOP TO BOTTOM.
5. Notebooks are saved to your Google Drive
6. Mount your Google Drive to have a direct access from a notebook to the files stored in the drive (this includes Team Drives).
7. If using Colab's virtual storage only, all the uploaded/stored files will get deleted when a runtime is recycled.

### Shell commands

The command `uname` displays the information about the system.

* **-a option:** It prints all the system information in the following order: Kernel name, network node hostname,
kernel release date, kernel version, machine hardware name, hardware platform, operating system
.

In [ ]:
!uname -a && cat /etc/*release

In [ ]:
!pwd

In [ ]:
!ls -la

In [ ]:
!pip list

## Wurlitzer for Numba (Colab + VS Code)

-	When working with Numba CUDA in notebooks, you may notice that:
    -	print() inside CPU jitted code or CUDA kernels can behave unexpectedly
    -	output from native code (C/C++/CUDA runtime) may not appear in the notebook cell output
    -	In Google Colab, this is common because the notebook captures Python stdout, but not always low-level stdout/stderr produced by compiled extensions.
-	Wurlitzer fixes this by capturing C-level stdout/stderr and forwarding it to the notebook output.

- Typical use-cases
	-	Seeing debug output from:
	    -	Numba-compiled code paths
	    -	CUDA driver/runtime messages
	    -	Getting consistent console logs when running a notebook from VS Code as a client (remote kernels / Jupyter integration).


#### How to Use It (Colab + VS Code Client)

- Install if necessary

    `pip -q install wurlitzer`

- Basic pattern: capture native output for a block


In [ ]:
from wurlitzer import sys_pipes
from numba import cuda

@cuda.jit
def hello_kernel():
    print("Hello from GPU!")
    
# launch with 1 block, 10 thread
print("Launching kernel...")

hello_kernel[1, 10]()       # type: ignore
with sys_pipes():           # captures C-level stdout/stderr
    cuda.synchronize()

# ▶️ CUDA tools...

**NVIDIA System Management Interface (nvidia-smi)**

The NVIDIA System Management Interface (**`nvidia-smi`**) is a command line utility, based on top of the NVIDIA Management Library (NVML), intended to aid in the **management** and **monitoring** of NVIDIA GPU devices.

This utility allows administrators to query GPU device state and with the appropriate privileges, permits administrators to modify GPU device state.  It is targeted at the TeslaTM, GRIDTM, QuadroTM and Titan X product, though limited support is also available on other NVIDIA GPUs.

For more details, please refer to the **`nvidia-smi`** documentation ([doc](http://developer.download.nvidia.com/compute/DCGM/docs/nvidia-smi-367.38.pdf))

For information on **Tesla T4** see:

In [ ]:
!nvidia-smi

**Numba code used as a CUDA sanity check**:
-   Imports Numba, a JIT compiler that accelerates Python code.
-	numba.cuda provides GPU (CUDA) support using NVIDIA GPUs.
	- ✔ Confirms NumPy and Numba are installed
	- ✔ Confirms CUDA drivers are visible
	- ✔ Confirms GPU compute capability
	- ✔ Helps debug environment issues before running GPU kernels

Probes the system for available CUDA-capable GPUs. Prints:
-	Number of GPUs
-	GPU names
-	Compute capability
-	Driver/runtime status


In [ ]:
# Check installed versions of numba and llvmlite
!pip list | grep numba 
!pip list | grep llvmlite

In [ ]:
import numpy as np
import numba
from numba import cuda
import warnings
warnings.filterwarnings("ignore")

print(np.__version__)
print(numba.__version__)

cuda.detect()



In [ ]:
# Suppress Numba deprecation and performance warnings
from numba.core.errors import NumbaDeprecationWarning, NumbaPerformanceWarning
import warnings

warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPerformanceWarning)

# ✅ Hello World!

**My first CUDA program: HelloFromGPU!**

CUDA kernel for Hello world...

In [ ]:
from numba import cuda

@cuda.jit
def hello_kernel():
    print("Hello from GPU!")
    
# launch with 1 block, 10 thread
print("Launching kernel...")
hello_kernel[1, 10]()    # type: ignore
cuda.synchronize()

# ✅ Parallel vector sum

In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def add_arrays_cuda(a, b, c):
    i = cuda.grid(1)
    if i < c.size:
        c[i] = a[i] + b[i]
    

# Example
n = 1000    # size of arrays = number of threads
a = np.ones(n, dtype=np.float32)
b = np.ones(n, dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array_like(a)

add_arrays_cuda[1, n](d_a, d_b, d_c)
cuda.synchronize()

c = d_c.copy_to_host()
print(c[0:10], c[-10:])

## ↘️ TODO...

**Check Fibonacci Membership** 

🔹 **Exercise Goal**

-   Given a vector $x$ of integers, build a CUDA kernel (Numba) that produces a vector $y$ such that:

$$
y_i =
\begin{cases}
1 & \text{if } x_i \text{ is a Fibonacci number} \\
0 & \text{otherwise}
\end{cases}
$$


🔹 **Input** 

- `x`: 1D array of integers (e.g., `np.int32`)
- `n = x.size`

🔹 **Output**

- `y`: 1D array of `np.uint8` or `np.int32`
- Same length as `x`


🔹 **Constraints**

- One GPU thread handles one element:
  $$
  i = \text{cuda.grid}(1)
  $$
- Must include a bounds check:
  $$
  i < n
  $$
- Avoid Python objects and dynamic allocation inside the kernel


🔹 **A classic property:**

> An integer $v \ge 0$ is Fibonacci **iff** one of these is a perfect square:
$$
5v^2 + 4 \quad \text{or} \quad 5v^2 - 4
$$


🔹 **Your Tasks**

1. Write a device function:
   - `is_perfect_square(m) -> bool`

2. Write a device function:
   - `is_fib(v) -> bool`

3. Write a kernel:
   - `fib_mask(x, y)` that sets `y[i] = 1` if `x[i]` is Fibonacci else `0`

4. Write host code to:
   - allocate/copy arrays to GPU
   - launch the kernel
   - copy results back and validate

🔹 **Expected result** 
-    for the first $n = 1000$ integers: [0,   1,   2,   3,   5,   8,  13,  21,  34,  55,  89, 144, 233, 377, 610, 987]

🔹 **Skeleton Code (Fill the TODOs)**

```{python}
    #| echo: true
    import numpy as np
    from numba import cuda
    import math

    @cuda.jit
    def fib_numbers(x,y):
        # TODO
        pass

    # input data
        # TODO

    # GPU memory allocation
        # TODO

    # Kernel launch
        # TODO

    # Copy back results & print indexes of Fibonacci numbers
        # TODO
```

## ➡️ Solution...

In [ ]:
## SOLUTION

import numpy as np
from numba import cuda
import math

@cuda.jit
def fib_numbers(x, y):
    i = cuda.grid(1)
    if i < x.size:
        # Use 64-bit intermediates to avoid overflow for typical int32 ranges
        vv = np.int64(x[i])
        t1 = np.int64(5) * vv * vv + np.int64(4)
        t2 = np.int64(5) * vv * vv - np.int64(4)
        
        # Check if either t1 or t2 is a perfect square
        r1 = int(math.sqrt(t1))  # sqrt returns float; safe for our chosen ranges
        r2 = int(math.sqrt(t2))  # sqrt returns float; safe for our chosen ranges
        if r1 * r1 == t1 or r2 * r2 == t2:
            y[i] = 1 
        else: 
            y[i] = 0


# input data
n = 1000
x = np.arange(0, n+1, dtype=np.int32)

# GPU memory allocation
d_x = cuda.to_device(x)
d_y = cuda.device_array(n, dtype=np.uint8)

# Kernel launch
threads = n
fib_numbers[1, threads](d_x, d_y)    # type: ignore
cuda.synchronize()  # wait for the kernel to finish

# Copy back results & print indexes of Fibonacci numbers
y = d_y.copy_to_host()
idx = np.nonzero(y)
print(idx[0])
